# ScholarBot Evaluation

Workspace for measuring and improving agent quality. Follows [EVAL_GUIDE.md](../docs/EVAL_GUIDE.md).

**Prerequisites:**
- Docker Compose running (MLflow at localhost:5000)
- `.env` with Foundry credentials

**Run with:** `uv run jupyter lab` from the project root

## One-Time Setup

In [1]:
import mlflow
import mlflow.anthropic  # noqa: F811
import dotenv

dotenv.load_dotenv()

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("scholarbot-eval")
mlflow.anthropic.autolog()  # type: ignore[attr-defined]

print(f"MLflow tracking: {mlflow.get_tracking_uri()}")

MLflow tracking: http://localhost:5000


### Prepare the dataset

Load DeepSearchQA and transform to MLflow eval schema. Run once — reuse across cycles.

In [2]:
from evaluation import load_deepsearchqa

test_cases = load_deepsearchqa(n=25, seed=42)

print(f"Loaded {len(test_cases)} test cases")
print(f"Categories: {set(tc['tags']['category'] for tc in test_cases)}")

Loaded 25 test cases
Categories: {'Politics & Government', 'Science', 'Geography', 'Other', 'Finance & Economics', 'Health', 'Education', 'Media & Entertainment', 'Arts'}


In [3]:
# Glance at a few examples
for tc in test_cases[:5]:
    print(f"\nQ: {tc['inputs']['question'][:150]}...")
    print(f"A: {tc['expectations']['answer']}")
    print(f"Type: {tc['expectations']['answer_type']}")
    print(f"Category: {tc['tags']['category']}")


Q: There is a type of instrument that has metal, gut, or horsehair and open strings, which has different versions still used for a few countries’ nationa...
A: Nyckelharpa, Hhardingfele
Type: Set Answer
Category: Arts

Q: According to Eurostat, which of the following countries saw a decrease of less than 45% in suicides involving railways between the years of 2014 and 2...
A: Germany, Belgium, UK, France, Bulgaria
Type: Set Answer
Category: Health

Q: According to the Council on Tall Buildings and Urban Habitat information on the tallest buildings in the world in 1990, from number 44 to 46 on the li...
A: New York City
Type: Single Answer
Category: Politics & Government

Q: Base your output on 1) the number of active Federal Firearms Licenses (FFLs) as of July 2021 according to Orchid Advisors, and 2) Firearm Mortality by...
A: Alabama
Type: Single Answer
Category: Politics & Government

Q: According to the IUCN red list, what are the scientific names of every plant and animal species

### Evaluation helpers

Variant-aware predict function and evaluation runner.

In [4]:
from evaluation import make_predict_fn

def evaluate_variant(variant: str, test_cases, scorers=None):
    """Generate traces and optionally score a variant. Returns the eval result."""
    predict_fn = make_predict_fn(variant)

    with mlflow.start_run(run_name=variant, tags={"variant": variant}):
        results = mlflow.genai.evaluate(
            data=test_cases,
            predict_fn=predict_fn,
            scorers=scorers or [],
        )
    return results

---

## Phase B: Establish Baseline

### B1. Generate traces for the baseline variant

Expect 5-15 minutes on 25 questions with Haiku.

In [5]:
baseline_result = evaluate_variant("v1-baseline", test_cases)
print(f"Generated {len(baseline_result.result_df)} traces")

2026/04/14 23:05:50 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/04/14 23:05:50 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating: |          | 0/0 [Elapsed: 00:00, Remaining: ?]

Generated 25 traces


In [6]:
baseline_result.result_df.head()

,trace_id,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-250dccb030918493b7854022c803de09,"{""info"": {""trace_id"": ""tr-250dccb030918493b785...",None,OK,1776229755376,84676,{'question': 'There is a type of instrument th...,"Based on my research, I can now solve this puz...",{'mlflow.source.git.commit': 'f53d4bc93c78f61f...,{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'JQ3MsDCRhJO3hUAiyAPeCQ==', 'spa...",[]
1,tr-e5ce6c9acab9de727263a63883f01c3a,"{""info"": {""trace_id"": ""tr-e5ce6c9acab9de727263...",None,OK,1776229755378,129572,"{'question': 'According to Eurostat, which of ...","Based on the Eurostat data, the countries that...",{'mlflow.source.git.commit': 'f53d4bc93c78f61f...,{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': '5c5smsq53nJyY6Y4g/AcOg==', 'spa...",[]
2,tr-6f1c9aa10a741959e11fd732259e28fa,"{""info"": {""trace_id"": ""tr-6f1c9aa10a741959e11f...",None,OK,1776229755380,53243,{'question': 'According to the Council on Tall...,"Based on my research, I can answer your questi...",{'mlflow.source.git.commit': 'f53d4bc93c78f61f...,{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'bxyaoQp0GVnhH9cyJZ4o+g==', 'spa...",[]
3,tr-5536018221620302c9d5fbd937944aa2,"{""info"": {""trace_id"": ""tr-5536018221620302c9d5...",None,OK,1776229755381,142505,{'question': 'Base your output on 1) the numbe...,"Based on my research, I can now provide you wi...",{'mlflow.source.git.commit': 'f53d4bc93c78f61f...,{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'VTYBgiFiAwLJ1fvZN5RKog==', 'spa...",[]
4,tr-951bb85def520584d8f35ec39fbc7079,"{""info"": {""trace_id"": ""tr-951bb85def520584d8f3...",None,OK,1776229755382,131241,"{'question': 'According to the IUCN red list, ...","Based on my research, I need to provide you wi...",{'mlflow.source.git.commit': 'f53d4bc93c78f61f...,{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'lRu4Xe9SBYTY817Dn7xweQ==', 'spa...",[]
